# pivotai — Fine-Tuned Model (`pivotai-ft`)
**Session 1 of 3** — Trains on Phase 1 synthetic data (4,750 records).

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `pivotai.zip` (full project) to Google Drive root
3. Add `HF_TOKEN` to Colab Secrets (🔑 icon in left sidebar) — write-access token from huggingface.co/settings/tokens
4. Run all cells in order

In [ ]:
# Cell 0 — Set memory env var FIRST (must run before any torch import)
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('✓ CUDA alloc config set')

In [ ]:
# Cell 1 — Mount Drive + unzip project
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os
if not os.path.exists('/content/travel_project'):
    with zipfile.ZipFile('/content/drive/MyDrive/pivotai.zip', 'r') as z:
        z.extractall('/content/')
    print('✓ Unzipped')
else:
    print('✓ Already extracted')

os.makedirs('/content/drive/MyDrive/pivotai_models', exist_ok=True)

In [ ]:
# Cell 2 — Add project to path + verify training files
import sys
sys.path.insert(0, '/content/travel_project')

TRAIN_FILE = '/content/travel_project/data/training/ft_train.jsonl'
VAL_FILE   = '/content/travel_project/data/training/ft_val.jsonl'

for f in [TRAIN_FILE, VAL_FILE]:
    lines = open(f).read().splitlines()
    print(f'{os.path.basename(f)}: {len(lines)} records')

In [ ]:
# Cell 3 — Install dependencies (~5 min)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl peft accelerate bitsandbytes -q
print('✓ Install complete')

In [ ]:
# Cell 4 — Load Llama 3.1 8B with 4-bit QLoRA
# max_seq_length=512 and r=8 are required to fit T4 (15GB VRAM)
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Meta-Llama-3.1-8B-bnb-4bit',
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('✓ Model loaded')

In [ ]:
# Cell 5 — Load dataset + filter sequences > 512 tokens
# Unsloth truncates silently but labels stay full length → shape mismatch in loss
# Pre-filtering avoids this entirely
from datasets import load_dataset

ALPACA_PROMPT = """### Instruction:\n{}\n\n### Input:\n{}\n\n### Response:\n{}"""
EOS = tokenizer.eos_token

def format_alpaca(examples):
    return {'text': [
        ALPACA_PROMPT.format(i, inp, out) + EOS
        for i, inp, out in zip(examples['instruction'], examples['input'], examples['output'])
    ]}

def filter_length(examples):
    lengths = tokenizer(examples['text'], truncation=False, return_length=True, padding=False)['length']
    return [l <= MAX_SEQ_LEN for l in lengths]

train_ds = load_dataset('json', data_files=TRAIN_FILE, split='train').map(format_alpaca, batched=True)
val_ds   = load_dataset('json', data_files=VAL_FILE,   split='train').map(format_alpaca, batched=True)

before = len(train_ds)
train_ds = train_ds.filter(filter_length, batched=True)
print(f'✓ Train: {before} → {len(train_ds)} (dropped {before - len(train_ds)} > {MAX_SEQ_LEN} tokens)')
print(f'✓ Val: {len(val_ds)}')

In [ ]:
# Cell 6 — Train (~2 hours on T4)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=20,
        fp16=True,
        logging_steps=50,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir='/content/pivotai_ft_checkpoints',
        save_strategy='no',
        report_to='none',
        dataloader_num_workers=0,
    ),
)

stats = trainer.train()
print(f'✓ Training complete — {stats.metrics["train_runtime"]:.0f}s ({stats.metrics["train_runtime"]/3600:.1f}h)')

In [ ]:
# Cell 7 — Export GGUF locally, copy to Drive, upload to HuggingFace
# IMPORTANT: save to local /content/ first, NOT Drive
# Unsloth reads the merged model back for GGUF conversion — Drive buffering causes 0-byte file errors
import shutil
from huggingface_hub import HfApi
from google.colab import userdata

HF_TOKEN   = userdata.get('HF_TOKEN')   # set in Colab Secrets (🔑 left sidebar)
HF_USERNAME = 'ishreyadev'

# Step 1: Convert to GGUF locally
model.save_pretrained_gguf('/content/pivotai_ft', tokenizer, quantization_method='q4_k_m')
print('✓ GGUF created')

# Step 2: Locate GGUF (Unsloth saves to {name}_gguf/Meta-Llama-3.1-8B.Q4_K_M.gguf)
gguf_src = '/content/pivotai_ft_gguf/Meta-Llama-3.1-8B.Q4_K_M.gguf'

# Step 3: Copy to Drive
gguf_dst = '/content/drive/MyDrive/pivotai_models/pivotai_ft.gguf'
shutil.copy(gguf_src, gguf_dst)
print(f'✓ Copied to Drive: {gguf_dst}')

# Step 4: Upload LoRA adapter to HF (~80MB, fast)
model.push_to_hub(f'{HF_USERNAME}/pivotai-ft-lora', token=HF_TOKEN, private=False)
tokenizer.push_to_hub(f'{HF_USERNAME}/pivotai-ft-lora', token=HF_TOKEN)
print(f'✓ LoRA → huggingface.co/{HF_USERNAME}/pivotai-ft-lora')

# Step 5: Upload GGUF to HF (~4.5GB, ~10 min)
api = HfApi()
api.create_repo(f'{HF_USERNAME}/pivotai-ft-gguf', token=HF_TOKEN, exist_ok=True)
api.upload_file(
    path_or_fileobj=gguf_src,
    path_in_repo='pivotai_ft.Q4_K_M.gguf',
    repo_id=f'{HF_USERNAME}/pivotai-ft-gguf',
    token=HF_TOKEN,
)
print(f'✓ GGUF → huggingface.co/{HF_USERNAME}/pivotai-ft-gguf')
print('\npivotai-ft COMPLETE')